# Lab run: train the Lab's models on a cloud GPU

The web app's **Lab** page packs everything about a model (its architecture, training settings and datasets) into
a **run spec**, and copies it for you. This notebook runs that spec on Colab or Kaggle and gives you a zip to
bring home.

1. **Use a GPU.** Colab: *Runtime → Change runtime type → T4 GPU* (free), or L4 / A100 with Colab Pro.
   Kaggle: *Settings → Accelerator → GPU*, and turn *Internet* on.
2. **Paste the spec** into the `spec` field below (on Kaggle, between the quotes).
3. **Run all** (*Runtime → Run all*). Datasets are downloaded and tokenized here, each model is trained the same
   way as the A/B tests (32 × 256-token updates, seed 42), evaluated, and packed into `ab-<stamp>.zip`.
4. **Bring it home.** On Colab the zip downloads by itself; on Kaggle it's in the *Output* panel. Put it in the
   project folder (or Downloads) and press **Import** in the Lab, or run `python3 import_run.py ab-<stamp>.zip`.


In [ ]:
# @title Get the code
import os, sys

REPO_URL = "https://github.com/a-ramakrishnan/tinystories-ARM-model.git"
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PLATFORM = "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local"

if IN_COLAB or IN_KAGGLE:
    # Kaggle keeps everything under /kaggle/working as the notebook's output, so the code and data go to /tmp
    # and only the final zip is saved there.
    CLONE_DIR = "/content/tinystories-ARM-model" if IN_COLAB else "/tmp/tinystories-ARM-model"
    if not os.path.exists(CLONE_DIR):
        !git clone -q {REPO_URL} {CLONE_DIR}
    else:
        !git -C {CLONE_DIR} pull -q   # already cloned in this session: get the latest code
    %cd {CLONE_DIR}
    !pip install -q -r requirements.txt

assert os.path.exists("full-model.py"), "Open this notebook from the tinystories-ARM-model folder."
PROJECT_DIR = os.getcwd()
OUTPUT_DIR = "/kaggle/working" if IN_KAGGLE else PROJECT_DIR   # where the results zip is written
print(f"Running on {PLATFORM}. Project folder: {PROJECT_DIR}")

In [ ]:
# @title Run the spec
spec = ""  # @param {type:"string"}

import torch
import ab_test
from IPython.display import Image, Markdown, display

if not spec.strip():
    raise SystemExit("Paste the run spec from the Lab into the spec field first.")
if not torch.cuda.is_available():
    print("No GPU: this will be very slow. Switch the runtime to a GPU first.")
else:
    print("GPU:", torch.cuda.get_device_name(0))
s = ab_test.decode_spec(spec)
summary, ab_dir, ab_zip = ab_test.run_spec(s, zip_dir=OUTPUT_DIR)
display(Markdown(open(os.path.join(ab_dir, "results.md")).read()))
display(Image(os.path.join(ab_dir, "losses.png")))
if IN_COLAB:
    from google.colab import files
    files.download(ab_zip)
else:
    print(f"Download {os.path.basename(ab_zip)} from the Output panel, then import it in the Lab.")
